In [ ]:
import pandas as pd
import re
from collections import defaultdict
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from huggingface_hub import login

# ----------- CONFIGURAZIONE MODELLO ------------------
#Codice per l'accesso al modello privato su Hugging Face

MODEL_NAME = "Qwen/Qwen3-8B"
DEVICE = -1  # Use CPU; change to 0 if using GPU

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)




/home/lucamariotti/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-03 18:19:11.769465: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748967551.874386   45424 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748967551.905280   45424 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-03 18:19:12.169501: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical op

Loading model...


Loading checkpoint shards:  40%|████      | 2/5 [00:14<00:21,  7.33s/it]

In [ ]:
# ----------- FUNZIONE BASE PER LLM ------------------
def _call_llm(prompt, max_new_tokens=10):
    output = llm(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.2,
        top_k=40,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )[0]['generated_text']
    return output[len(prompt):].strip()

# ----------- PROMPT CON FEW-SHOT EXAMPLES ------------

def build_validation_prompt(sentence, head, tail, relation):
    return f"""Imagine you are a human annotator tasked with determining if a sentence clearly expresses a relation between two entities.
You will be given a sentence, a head entity, a tail entity, and a proposed relation. Your goal is to decide if the sentence clearly expresses the relation between the head and tail.
Examples:
Sentence: "Barack Obama was born in Hawaii."
Head: Barack Obama
Tail: Hawaii
Relation: place_of_birth 
Your answer: yes

Sentence: "Einstein wrote a paper on relativity."
Head: Einstein
Tail: relativity
Relation: declare_war
Your answer: no

Now evaluate the following:
Sentence: {sentence}
Head: {head}
Tail: {tail}
Relation: {relation}

Question: Does the sentence clearly express the relation between Head and Tail?
Answer with only 'yes' or 'no' — no explanations or additional text.
"""


def build_correction_prompt(sentence, head, tail):
    return f"""Imagine you are a human annotator tasked with identifying the most accurate relation between two entities in a sentence.
You will be given a sentence, a head entity, and a tail entity. Your goal is to determine the most appropriate relation between the head and tail based on the context of the sentence.
Examples:
Sentence: "Barack Obama was born in Hawaii."
Head: Barack Obama
Tail: Hawaii
Your answer: place_of_birth

Sentence: "Steve Jobs founded Apple."
Head: Steve Jobs
Tail: Apple
Your answer: founder_of

Now evaluate the following:
Sentence: {sentence}
Head: {head}
Tail: {tail}

Question: What is the most accurate relation between Head and Tail?
Answer with only the relation name (e.g., founder_of) or 'none'. Do not include any explanation or additional text.
"""


# ----------- FUNZIONI DI VALIDAZIONE E CORREZIONE ----

def validate_relation(sentence, head, tail, relation):
    prompt = build_validation_prompt(sentence, head, tail, relation)
    raw_output = _call_llm(prompt, max_new_tokens=5).lower()
    print(f"[DEBUG] Validation output: {raw_output}")
    
    if 'yes' in raw_output:
        return 'yes'
    elif 'no' in raw_output:
        return 'no'
    else:
        print(f"[WARNING] Unexpected validation output: {raw_output}")
        return 'no'

def correct_relation(sentence, head, tail):
    prompt = build_correction_prompt(sentence, head, tail)
    correction = _call_llm(prompt, max_new_tokens=150).strip().lower()
    print(f"[DEBUG] Correction output: {correction}")
    return correction

# ----------- FUNZIONE PRINCIPALE ---------------------

def process_dataset(df):
    corrected_rows = []
    stats = defaultdict(int)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        sent, h, t, rel = row['sentence'], row['head'], row['tail'], row['relation']
        validation = validate_relation(sent, h, t, rel)

        if validation == "no":
            correction = correct_relation(sent, h, t)
            if correction not in ("none", "", "unknown"):
                corrected_rows.append({
                    "sentence": sent,
                    "head": h,
                    "tail": t,
                    "relation": correction
                })
                stats["corrected"] += 1
            else:
                stats["skipped"] += 1
        else:
            corrected_rows.append({
                "sentence": sent,
                "head": h,
                "tail": t,
                "relation": rel
            })
            stats["kept"] += 1

    cleaned_df = pd.DataFrame(corrected_rows)
    return cleaned_df, stats


In [ ]:
# ----------- USO DI ESEMPIO --------------------------

if __name__ == "__main__":
    data = {
        "sentence": [
            "Barack Obama was born in Hawaii.",
            "Steve Jobs founded Apple in 1976.",
            "Einstein wrote a paper on relativity.",
            "The United Nations has its headquarters in New York.",
        ],
        "head": ["Barack Obama", "Steve Jobs", "Einstein", "United Nations"],
        "tail": ["Hawaii", "Apple", "relativity", "New York"],
        "relation": ["place_of_birth", "founder_of", "declare_war", "located_in"]
    }

    df = pd.DataFrame(data)
    cleaned_df, stats = process_dataset(df)

    print("\n--- Dataset pulito ---")
    print(cleaned_df)

    print("\n--- Statistiche ---")
    for k, v in stats.items():
        print(f"{k}: {v}")

 25%|██▌       | 1/4 [00:11<00:34, 11.63s/it]

[DEBUG] Validation output: answer:

yes
okay


 50%|█████     | 2/4 [00:27<00:28, 14.38s/it]

[DEBUG] Validation output: answer:

yes
okay
[DEBUG] Validation output: answer:

no
the


 75%|███████▌  | 3/4 [05:20<02:21, 141.43s/it]

[DEBUG] Correction output: answer:
okay, let's see. the sentence is "einstein wrote a paper on relativity." the head entity is einstein, and the tail is relativity. i need to figure out the relation between them.

first, the action here is "wrote a paper on." so einstein is the author, and the paper is about relativity. the tail entity is "relativity," which is the topic of the paper. so the relation isn't about authorship of a paper, but rather the subject matter. 

in the examples given earlier, like "steve jobs founded apple," the relation was "founder_of." in the first example, "place_of_birth" was correct. so here, einstein wrote a paper on relativity
